# 03_Feature_Engineering

Notebook nay tap trung vao feature engineering theo dung huong pipeline cho tung nhom bai toan MLlib, su dung du lieu parquet tu notebook preprocessing.

## ROADMAP

- C1 Setup: Khoi tao Spark, load parquet theo layer silver/gold.
- C2 Data Contract: Kiem tra grain, null va input contract cho tung nhanh.
- C3 Classification FE: StringIndexer + OHE + TF-IDF/Word2Vec + ChiSqSelector.
- C4 Regression FE: Tabular pipeline cho du bao order value/freight.
- C5 Clustering FE: RFM + hanh vi + scaling cho customer segmentation.
- C6 ALS Prep: Tao user_idx/item_idx/rating cho recommendation.
- C7 FP-Growth Prep: Chuan hoa basket items cho association rules.
- C8 Materialization: Luu feature outputs va model-prep artifacts.


**[C1-1] Import thư viện**

- Thư viện cơ bản: `pathlib`, `os`, `sys`, `warnings`, `pandas`, `matplotlib`.
- PySpark core: `SparkSession`, `DataFrame`, `functions as F`.
- **MLlib feature transformers** — toàn bộ pipeline FE dùng các class sau:

| Class              | Công dụng                                       |
| ------------------ | ----------------------------------------------- |
| `StringIndexer`    | Encode cột string → số nguyên theo tần suất     |
| `OneHotEncoder`    | Encode index → vector nhị phân (OHE)            |
| `RegexTokenizer`   | Tách văn bản thành token theo regex             |
| `StopWordsRemover` | Loại bỏ stop words tiếng Anh                    |
| `HashingTF`        | Tạo TF vector dùng hashing trick                |
| `IDF`              | Tính trọng số IDF để tạo TF-IDF                 |
| `Word2Vec`         | Học word embedding phân tán                     |
| `VectorAssembler`  | Gom nhiều cột → một vector feature              |
| `ChiSqSelector`    | Chọn top-K features theo Chi-Square test        |
| `StandardScaler`   | Chuẩn hóa feature theo độ lệch chuẩn            |
| `Pipeline`         | Kết nối các stage thành một workflow thống nhất |


In [24]:
# Command 26 - Setup environment va load parquet dau vao
from pathlib import Path
import os
import sys
import warnings

import pandas as pd
import matplotlib.pyplot as plt

from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F

from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    ChiSqSelector,
    HashingTF,
    IDF,
    OneHotEncoder,
    RegexTokenizer,
    StandardScaler,
    StopWordsRemover,
    StringIndexer,
    VectorAssembler,
    Word2Vec,
)



**[C1-2] Cấu hình môi trường & khởi tạo SparkSession**

- `warnings.filterwarnings('ignore')`: tắt warning không liên quan.
- `pd.set_option('display.max_columns', 200)`: hiển thị đầy đủ cột khi `.to_string()`.
- **SparkSession config**:
  - `local[2]` + `spark.driver.memory=4g`: chạy local 2 thread, 4GB RAM.
  - `spark.sql.shuffle.partitions=16`: giới hạn partition khi shuffle (dataset nhỏ/vừa).
  - `spark.sql.session.timeZone=UTC`: chuẩn hóa timezone cho timestamp.
  - Đặt `PYSPARK_PYTHON` = interpreter hiện tại để tránh version mismatch.


In [25]:
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
plt.style.use("seaborn-v0_8-whitegrid")

python_exec = sys.executable
os.environ["PYSPARK_PYTHON"] = python_exec
os.environ["PYSPARK_DRIVER_PYTHON"] = python_exec

active_spark = SparkSession.getActiveSession()
if active_spark is not None:
    active_spark.stop()

spark = (
    SparkSession.builder
    .appName("Nhom03_Olist_FeatureEngineering") # type: ignore
    .master("local[2]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "16")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.python.worker.reuse", "false")
    .config("spark.pyspark.python", python_exec)
    .config("spark.pyspark.driver.python", python_exec)
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")


**[C1-3] Thiết lập đường dẫn & helper function**

- Xây dựng cây thư mục output: `data/processed/features/` và `data/processed/features/models/`.
- `read_parquet_checked(path)`: wrapper đọc parquet với kiểm tra tồn tại trước;
  nếu thiếu file sẽ raise `FileNotFoundError` rõ ràng thay vì lỗi Spark mập mờ.
- Separation of concern: đường dẫn `GOLD_DIR` (input) và `FEATURE_DIR` (output)
  tách biệt để không ghi đè dữ liệu gốc.


In [26]:

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
SILVER_DIR = PROCESSED_DIR / "silver"
GOLD_DIR = PROCESSED_DIR / "gold"
FEATURE_DIR = PROCESSED_DIR / "features"
MODELS_DIR = FEATURE_DIR / "models"
for output_dir in [FEATURE_DIR, MODELS_DIR]:
    output_dir.mkdir(parents=True, exist_ok=True)

def read_parquet_checked(path: Path) -> DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Missing parquet path: {path}")
    return spark.read.parquet(str(path))


**[C1-4] Load gold splits và silver review từ parquet**

- Load các `gold` dataset theo cấu trúc mới: classification/regression/clustering đã tách
  sẵn `train`/`val`/`test`; ALS và FP-Growth vẫn đọc từ base parquet.
- In `input_summary_pdf` để xác nhận row count của từng dataset trước khi build pipeline.
- **Tại sao load riêng `order_reviews` silver?** Bảng gold classification không giữ
  raw text review — text được join lại ở bước FE để tránh materialization quá nặng.


In [27]:
classification_train_base_df = read_parquet_checked(GOLD_DIR / "classification_base_train")
classification_val_base_df = read_parquet_checked(GOLD_DIR / "classification_base_val")
classification_test_base_df = read_parquet_checked(GOLD_DIR / "classification_base_test")
classification_base_df = classification_train_base_df.unionByName(classification_val_base_df).unionByName(classification_test_base_df)

regression_train_base_df = read_parquet_checked(GOLD_DIR / "regression_base_train")
regression_val_base_df = read_parquet_checked(GOLD_DIR / "regression_base_val")
regression_test_base_df = read_parquet_checked(GOLD_DIR / "regression_base_test")
regression_base_df = regression_train_base_df.unionByName(regression_val_base_df).unionByName(regression_test_base_df)

clustering_train_base_df = read_parquet_checked(GOLD_DIR / "clustering_base_train")
clustering_val_base_df = read_parquet_checked(GOLD_DIR / "clustering_base_val")
clustering_test_base_df = read_parquet_checked(GOLD_DIR / "clustering_base_test")
clustering_base_df = clustering_train_base_df.unionByName(clustering_val_base_df).unionByName(clustering_test_base_df)
als_base_df = read_parquet_checked(GOLD_DIR / "als_base")
fpgrowth_base_df = read_parquet_checked(GOLD_DIR / "fpgrowth_base")
order_reviews_silver_df = read_parquet_checked(SILVER_DIR / "order_reviews")

input_summary_pdf = pd.DataFrame(
    [
        ("classification_base_train", classification_train_base_df.count()),
        ("classification_base_val", classification_val_base_df.count()),
        ("classification_base_test", classification_test_base_df.count()),
        ("regression_base_train", regression_train_base_df.count()),
        ("regression_base_val", regression_val_base_df.count()),
        ("regression_base_test", regression_test_base_df.count()),
        ("clustering_base_train", clustering_train_base_df.count()),
        ("clustering_base_val", clustering_val_base_df.count()),
        ("clustering_base_test", clustering_test_base_df.count()),
        ("als_base", als_base_df.count()),
        ("fpgrowth_base", fpgrowth_base_df.count()),
    ],
    columns=["dataset_name", "row_count"],
).sort_values("dataset_name").reset_index(drop=True)
print(input_summary_pdf.to_string(index=False))

print(f"Spark version: {spark.version}")
print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"GOLD_DIR: {GOLD_DIR}")


             dataset_name  row_count
                 als_base     101987
 classification_base_test      14734
classification_base_train      68887
  classification_base_val      14546
     clustering_base_test      13967
    clustering_base_train      65592
      clustering_base_val      13799
            fpgrowth_base      98666
     regression_base_test      14761
    regression_base_train      69294
      regression_base_val      14611
Spark version: 3.5.8
PROJECT_ROOT: /Users/thuannguyen/Downloads/Vannhi/Nhom03_BigDataCuoiKy_Update_01
GOLD_DIR: /Users/thuannguyen/Downloads/Vannhi/Nhom03_BigDataCuoiKy_Update_01/data/processed/gold


**[C2-1] Helper function `null_ratio`**

- Hàm tính tỉ lệ null của một cột trong DataFrame: `null_count / total_rows`.
- Trả về `0.0` nếu DataFrame rỗng (tránh chia cho 0).
- Được dùng lặp lại cho cột critical của từng base dataset.


In [28]:
# Command 27 - Data contract audit cho cac base datasets
def null_ratio(df: DataFrame, col_name: str) -> float:
    total = df.count()
    if total == 0:
        return 0.0
    return df.filter(F.col(col_name).isNull()).count() / total


**[C2-2] Data Contract Audit — kiểm tra 5 datasets đầu vào**

Với mỗi dataset, audit 3 chỉ số:

- `row_count`: tổng số dòng — so sánh với preprocessing output để phát hiện mất mát.
- `distinct_key_count`: số key distinct (order_id / customer_unique_id / user+item pair).
  Nếu `row_count > distinct_key_count` → grain bị sai, cần quay lại preprocessing.
- `critical_null_ratio`: tỉ lệ null ở cột label/target chính —
  tỉ lệ cao có thể làm model bị bias hoặc cần chiến lược fill trước khi train.


In [29]:

regression_audit_col = "order_item_value" if "order_item_value" in regression_base_df.columns else "order_gmv"

contract_rows = [
    (
        "classification_base",
        classification_base_df.count(),
        classification_base_df.select("order_id").distinct().count(),
        round(null_ratio(classification_base_df, "is_low_review"), 4),
    ),
    (
        "regression_base",
        regression_base_df.count(),
        regression_base_df.select("order_id").distinct().count(),
        round(null_ratio(regression_base_df, regression_audit_col), 4),
    ),
    (
        "clustering_base",
        clustering_base_df.count(),
        clustering_base_df.select("customer_unique_id").distinct().count(),
        round(null_ratio(clustering_base_df, "monetary_value"), 4),
    ),
    (
        "als_base",
        als_base_df.count(),
        als_base_df.select("customer_unique_id", "product_id").distinct().count(),
        round(null_ratio(als_base_df, "implicit_rating"), 4),
    ),
    (
        "fpgrowth_base",
        fpgrowth_base_df.count(),
        fpgrowth_base_df.select("order_id").distinct().count(),
        round(null_ratio(fpgrowth_base_df, "items"), 4),
    ),
]

contract_checks_pdf = pd.DataFrame(
    contract_rows,
    columns=["dataset_name", "row_count", "distinct_key_count", "critical_null_ratio"],
).sort_values("dataset_name").reset_index(drop=True)
print(contract_checks_pdf.to_string(index=False))


       dataset_name  row_count  distinct_key_count  critical_null_ratio
           als_base     101987              101987                  0.0
classification_base      98167               98167                  0.0
    clustering_base      93358               93358                  0.0
      fpgrowth_base      98666               98666                  0.0
    regression_base      98666               98666                  0.0


**[C3-1] Chuẩn bị dữ liệu cho Classification — join text & tạo label**

- `TEXT_METHOD = 'tfidf'`: chọn phương pháp vectorize text (`'tfidf'` hoặc `'word2vec'`).
  Có thể đổi sang `'word2vec'` để thử nghiệm embedding phân tán.
- Tự động phát hiện tên cột text review (`review_comment_message_filled` nếu tồn tại).
- Mặc định chạy chế độ **early-risk**: không join review text để tránh leakage.
  (Có thể bật `USE_REVIEW_TEXT_FOR_CLASSIFICATION=True` cho bài toán sentiment hậu-review).
- `label = is_low_review.cast('int')` → target nhị phân: `1` nếu review_score ≤ 3, `0` nếu không.


In [30]:
# Command 28 - Classification feature engineering pipeline
TEXT_METHOD = "tfidf"  # "tfidf" hoac "word2vec"
USE_REVIEW_TEXT_FOR_CLASSIFICATION = False  # Early-risk mode: tranh leakage
USE_CHI_SQ_SELECTOR = False  # ChiSq yeu cau input categorical/discretized

review_text_col = "review_comment_message_filled" if "review_comment_message_filled" in order_reviews_silver_df.columns else "review_comment_message"

def prepare_cls_df(base_df: DataFrame) -> DataFrame:
    cls_local = base_df.withColumn("label", F.col("is_low_review").cast("int"))
    if USE_REVIEW_TEXT_FOR_CLASSIFICATION:
        cls_local = (
            cls_local
            .join(order_reviews_silver_df.select("order_id", F.coalesce(F.col(review_text_col), F.lit("")).alias("review_text")), on="order_id", how="left")
            .withColumn("review_text", F.coalesce(F.col("review_text"), F.lit("")))
        )
    return cls_local

cls_train_df = prepare_cls_df(classification_train_base_df)
cls_val_df = prepare_cls_df(classification_val_base_df)
cls_test_df = prepare_cls_df(classification_test_base_df)
cls_df = prepare_cls_df(classification_base_df)
print(f"classification upstream split rows -> train: {cls_train_df.count():,} | val: {cls_val_df.count():,} | test: {cls_test_df.count():,}")


classification upstream split rows -> train: 68,887 | val: 14,546 | test: 14,734


**[C3-2] Chuẩn hóa numeric features**

- `numeric_cols_cls`: danh sách 7 cột số tabular cho classification.
- Vòng `for`: ép kiểu về `double` và fill null bằng `0.0` bằng `F.coalesce()`.
  Điều này đảm bảo `VectorAssembler` không báo lỗi do null hay kiểu sai.
- Các cột được chọn có ý nghĩa nghiệp vụ rõ ràng cho early-risk và đã loại
  các biến outcome/post-order như `order_gmv`, `payment_total_value`, review fields.


In [31]:

# Map the required feature columns to available columns in your DataFrame
# If a column does not exist, fill with 0.0

numeric_cls_map = {
    "items_per_order": "order_item_count",
    "unique_products_per_order": None,  # TODO: compute if possible
    "unique_categories_per_order": None,  # TODO: compute if possible
    "max_payment_installments": "payment_installments_max",
    "same_state_any_seller_flag": None,  # TODO: compute if possible
}

for c, src in numeric_cls_map.items():
    if src and src in cls_train_df.columns:
        cls_train_df = cls_train_df.withColumn(c, F.coalesce(F.col(src).cast("double"), F.lit(0.0)))
        cls_val_df = cls_val_df.withColumn(c, F.coalesce(F.col(src).cast("double"), F.lit(0.0)))
        cls_test_df = cls_test_df.withColumn(c, F.coalesce(F.col(src).cast("double"), F.lit(0.0)))
        cls_df = cls_df.withColumn(c, F.coalesce(F.col(src).cast("double"), F.lit(0.0)))
    else:
        cls_train_df = cls_train_df.withColumn(c, F.lit(0.0))
        cls_val_df = cls_val_df.withColumn(c, F.lit(0.0))
        cls_test_df = cls_test_df.withColumn(c, F.lit(0.0))
        cls_df = cls_df.withColumn(c, F.lit(0.0))


**[C3-3] Định nghĩa các stage Categorical & Text**

- **Categorical branch** (`customer_state`):
  - `StringIndexer`: encode state string → index số theo tần suất (`handleInvalid='keep'` → unknown state → index riêng).
  - `OneHotEncoder`: index → vector nhị phân OHE (tránh ordinal relationship giả tạo).
- **Text branch** (review text):
  - `RegexTokenizer(pattern='[^A-Za-z0-9]+', minTokenLength=2)`: tách token, bỏ ký tự đặc biệt và token quá ngắn.
  - `StopWordsRemover`: loại bỏ stop words tiếng Anh để giảm nhiễu và chiều.
- **text_stages** (if/else theo `TEXT_METHOD`):
  - `'tfidf'` → `HashingTF(numFeatures=2^14=16384)` + `IDF` → vector TF-IDF sparse.
  - `'word2vec'` → `Word2Vec(vectorSize=100, minCount=2)` → vector dense 100 chiều.
  - Cả hai đều output ra cột `text_features` nhất quán cho stage tiếp theo.


In [32]:

cat_indexer_cls = StringIndexer(inputCol="customer_state", outputCol="customer_state_idx", handleInvalid="keep")
cat_encoder_cls = OneHotEncoder(inputCols=["customer_state_idx"], outputCols=["customer_state_ohe"], handleInvalid="keep")

numeric_cols_cls = list(numeric_cls_map.keys())

if USE_REVIEW_TEXT_FOR_CLASSIFICATION:
    tokenizer_cls = RegexTokenizer(inputCol="review_text", outputCol="tokens", pattern="[^A-Za-z0-9]+", minTokenLength=2)
    stop_rm_cls = StopWordsRemover(inputCol="tokens", outputCol="tokens_clean")
    if TEXT_METHOD == "word2vec":
        text_stages = [Word2Vec(inputCol="tokens_clean", outputCol="text_features", vectorSize=100, minCount=2)]
    else:
        text_stages = [
            HashingTF(inputCol="tokens_clean", outputCol="text_tf", numFeatures=1 << 14),
            IDF(inputCol="text_tf", outputCol="text_features"),
        ]
    cls_feature_inputs = numeric_cols_cls + ["customer_state_ohe", "text_features"]
    cls_pre_stages = [cat_indexer_cls, cat_encoder_cls, tokenizer_cls, stop_rm_cls] + text_stages
else:
    text_stages = []
    cls_feature_inputs = numeric_cols_cls + ["customer_state_ohe"]
    cls_pre_stages = [cat_indexer_cls, cat_encoder_cls]


**[C3-4] Assemble, ChiSqSelector, Scale & Fit Pipeline**

- `VectorAssembler`: gom 7 numeric + OHE state + text_features → `raw_features` (vector dày).
- `ChiSqSelector(numTopFeatures=300)`: chọn 300 feature có khả năng phân biệt label cao nhất
  theo Chi-Square test → giảm chiều, giảm overfitting.
- `StandardScaler(withMean=False, withStd=True)`: chuẩn hóa theo độ lệch chuẩn
  (`withMean=False` vì sparse vector không hỗ trợ centering).
- **Try/Except fallback**: nếu `ChiSqSelector` thất bại (ví dụ dữ liệu quá ít),
  pipeline tự động chạy lại không có selector — đảm bảo notebook không crash.
- Output cuối: DataFrame với 3 cột `(order_id, label, features)`.


In [33]:

assembler_cls = VectorAssembler(
    inputCols=cls_feature_inputs,
    outputCol="raw_features",
    handleInvalid="keep",
)
selector_cls = ChiSqSelector(featuresCol="raw_features", outputCol="selected_features", labelCol="label", numTopFeatures=300)
scaler_cls_selected = StandardScaler(inputCol="selected_features", outputCol="features", withMean=False, withStd=True)
scaler_cls_raw = StandardScaler(inputCol="raw_features", outputCol="features", withMean=False, withStd=True)

if USE_CHI_SQ_SELECTOR:
    print("ChiSqSelector mode: ENABLED")
    classification_pipeline = Pipeline(stages=cls_pre_stages + [assembler_cls, selector_cls, scaler_cls_selected])
else:
    print("ChiSqSelector mode: DISABLED (explicit, no silent fallback)")
    classification_pipeline = Pipeline(stages=cls_pre_stages + [assembler_cls, scaler_cls_raw])

classification_pipeline_model = classification_pipeline.fit(cls_train_df)
classification_train_df = classification_pipeline_model.transform(cls_train_df).select("order_id", "label", "features")
classification_val_df = classification_pipeline_model.transform(cls_val_df).select("order_id", "label", "features")
classification_test_df = classification_pipeline_model.transform(cls_test_df).select("order_id", "label", "features")
classification_fe_df = classification_pipeline_model.transform(cls_df).select("order_id", "label", "features")

classification_fe_df.show(5, truncate=False)
print(f"classification_fe rows: {classification_fe_df.count():,}")
print(f"classification split rows -> train: {classification_train_df.count():,} | val: {classification_val_df.count():,} | test: {classification_test_df.count():,}")


ChiSqSelector mode: DISABLED (explicit, no silent fallback)
+--------------------------------+-----+-----------------------------+
|order_id                        |label|features                     |
+--------------------------------+-----+-----------------------------+
|00018f77f2f0320c557190d7a144bdd3|0    |(33,[5],[2.0265098877733685])|
|00024acbcdf0a6daa1e931b038114c75|0    |(33,[5],[2.0265098877733685])|
|00048cc3ae777c65dbb7d2a0634bc1ea|0    |(33,[7],[3.1059575501206065])|
|0005a1a1728c9d785b8e2b08b904576c|1    |(33,[5],[2.0265098877733685])|
|000c3e6612759851cc3cbb4b83257986|0    |(33,[5],[2.0265098877733685])|
+--------------------------------+-----+-----------------------------+
only showing top 5 rows

classification_fe rows: 98,167
classification split rows -> train: 68,887 | val: 14,546 | test: 14,734


In [34]:
classification_fe_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- label: integer (nullable = true)
 |-- features: vector (nullable = true)



**[C4-1] Chuẩn bị dữ liệu Regression — label & numeric features**

- `label = order_gmv.cast('double')`: target liên tục (Gross Merchandise Value của đơn hàng).
- `numeric_cols_reg`: 6 features tabular, không trùng với `order_gmv` (tránh leakage).
  - `avg_product_weight_g`: đại diện cho độ phức tạp logistics.
  - `same_state_any_seller_flag`: giao hàng nội/ngoại tỉnh ảnh hưởng đến freight.
- Chuẩn hóa kiểu double + fill null = 0.0 tương tự classification.


In [35]:
# Command 29 - Regression feature engineering pipeline
# Normalize regression label source to avoid unresolved column errors
label_source_col = "order_item_value" if "order_item_value" in regression_train_base_df.columns else "order_gmv"

regression_train_base_df = regression_train_base_df.withColumn("order_item_value", F.col(label_source_col).cast("double"))
regression_val_base_df = regression_val_base_df.withColumn("order_item_value", F.col(label_source_col).cast("double"))
regression_test_base_df = regression_test_base_df.withColumn("order_item_value", F.col(label_source_col).cast("double"))
regression_base_df = regression_base_df.withColumn("order_item_value", F.col(label_source_col).cast("double"))

reg_train_df = regression_train_base_df.withColumn("label", F.col("order_item_value").cast("double"))
reg_val_df = regression_val_base_df.withColumn("label", F.col("order_item_value").cast("double"))
reg_test_df = regression_test_base_df.withColumn("label", F.col("order_item_value").cast("double"))
reg_df = regression_base_df.withColumn("label", F.col("order_item_value").cast("double"))

numeric_reg_map = {
    "items_per_order": "order_item_count",
    "unique_products_per_order": None,  # TODO: compute if possible
    "unique_categories_per_order": None,  # TODO: compute if possible
    "avg_product_weight_g": "avg_product_weight_g",
    "max_payment_installments": "payment_installments_max",
    "same_state_any_seller_flag": None,  # TODO: compute if possible
}

for c, src in numeric_reg_map.items():
    if src and src in reg_train_df.columns:
        reg_train_df = reg_train_df.withColumn(c, F.coalesce(F.col(src).cast("double"), F.lit(0.0)))
        reg_val_df = reg_val_df.withColumn(c, F.coalesce(F.col(src).cast("double"), F.lit(0.0)))
        reg_test_df = reg_test_df.withColumn(c, F.coalesce(F.col(src).cast("double"), F.lit(0.0)))
        reg_df = reg_df.withColumn(c, F.coalesce(F.col(src).cast("double"), F.lit(0.0)))
    else:
        reg_train_df = reg_train_df.withColumn(c, F.lit(0.0))
        reg_val_df = reg_val_df.withColumn(c, F.lit(0.0))
        reg_test_df = reg_test_df.withColumn(c, F.lit(0.0))
        reg_df = reg_df.withColumn(c, F.lit(0.0))

numeric_cols_reg = list(numeric_reg_map.keys())


**[C4-2] Pipeline Regression — StringIndexer → OHE → Assemble → Scale → Fit**

- Pipeline thuần tabular (không có text): `StringIndexer` → `OneHotEncoder` → `VectorAssembler` → `StandardScaler`.
- `order_freight_value` được giữ lại trong output để có thể dùng làm target thay thế
  (ví dụ: so sánh model dự báo GMV vs model dự báo freight riêng).
- Pipeline này tương thích với: `LinearRegression`, `DecisionTreeRegressor`, `RandomForestRegressor`.


In [36]:

state_indexer_reg = StringIndexer(inputCol="customer_state", outputCol="customer_state_idx", handleInvalid="keep")
state_encoder_reg = OneHotEncoder(inputCols=["customer_state_idx"], outputCols=["customer_state_ohe"], handleInvalid="keep")
assembler_reg = VectorAssembler(inputCols=numeric_cols_reg + ["customer_state_ohe"], outputCol="raw_features", handleInvalid="keep")
scaler_reg = StandardScaler(inputCol="raw_features", outputCol="features", withMean=False, withStd=True)

regression_pipeline = Pipeline(stages=[state_indexer_reg, state_encoder_reg, assembler_reg, scaler_reg])
regression_pipeline_model = regression_pipeline.fit(reg_train_df)
regression_train_df = regression_pipeline_model.transform(reg_train_df).select("order_id", "label", "order_freight_value", "features")
regression_val_df = regression_pipeline_model.transform(reg_val_df).select("order_id", "label", "order_freight_value", "features")
regression_test_df = regression_pipeline_model.transform(reg_test_df).select("order_id", "label", "order_freight_value", "features")
regression_fe_df = regression_pipeline_model.transform(reg_df).select("order_id", "label", "order_freight_value", "features")
regression_fe_df.show(5, truncate=False)
print(f"regression_fe rows: {regression_fe_df.count():,}")
print(f"regression split rows -> train: {regression_train_df.count():,} | val: {regression_val_df.count():,} | test: {regression_test_df.count():,}")


+--------------------------------+------------------+-------------------+---------------------------------------------------+
|order_id                        |label             |order_freight_value|features                                           |
+--------------------------------+------------------+-------------------+---------------------------------------------------+
|00018f77f2f0320c557190d7a144bdd3|259.83            |19.93              |(34,[3,6],[7.9302961659979925,2.0267301206802304]) |
|00024acbcdf0a6daa1e931b038114c75|25.78             |12.79              |(34,[3,6],[0.05286864110665328,2.0267301206802304])|
|00048cc3ae777c65dbb7d2a0634bc1ea|34.589999999999996|12.69              |(34,[3,8],[0.11895444248996988,3.1013064274996114])|
|00054e8431b9d7675808bcb819fb4a32|31.75             |11.85              |(34,[3,6],[0.05286864110665328,2.0267301206802304])|
|0008288aa423d2a3f00fcb17cd7d8719|126.53999999999999|26.74              |(34,[3,6],[0.4361662891298896,2.0267301206802

**[C5] Pipeline Clustering — RFM + Behavior → Scale → Features**

- **Grain**: customer-level (mỗi dòng = 1 `customer_unique_id`).
- `numeric_cols_cluster` = 4 chiều RFM + hành vi:

| Feature               | Ý nghĩa                                        |
| --------------------- | ---------------------------------------------- |
| `recency_days`        | Số ngày kể từ lần mua gần nhất (thấp = active) |
| `frequency_orders`    | Số đơn hàng đã đặt                             |
| `monetary_value`      | Tổng GMV đã giao thành công                    |
| `avg_items_per_order` | Hành vi chọn hàng trong giỏ                    |

- `customer_state` được OHE để thêm chiều địa lý.
- `StandardScaler`: **bắt buộc** cho clustering vì RFM có đơn vị và biên độ rất khác nhau
  (recency=số ngày, monetary=BRL). Không scale sẽ khiến monetary thống trị cluster.
- Output: `(customer_unique_id, features)` — sẵn sàng cho KMeans / BisectingKMeans / GaussianMixture.


In [37]:
# Command 30 - Clustering feature engineering pipeline

def prepare_cluster_df(base_df: DataFrame) -> DataFrame:
    if "customer_state" in base_df.columns:
        cluster_local = base_df
    else:
        cluster_local = base_df.join(
            classification_base_df.select("customer_unique_id", "customer_state").dropDuplicates(["customer_unique_id"]),
            on="customer_unique_id",
            how="left"
        )

    for c in numeric_cols_cluster:
        cluster_local = cluster_local.withColumn(c, F.coalesce(F.col(c).cast("double"), F.lit(0.0)))
    return cluster_local

numeric_cols_cluster = ["recency_days", "frequency_orders", "monetary_value", "avg_items_per_order"]
cluster_train_df = prepare_cluster_df(clustering_train_base_df)
cluster_val_df = prepare_cluster_df(clustering_val_base_df)
cluster_test_df = prepare_cluster_df(clustering_test_base_df)
cluster_df = prepare_cluster_df(clustering_base_df)

state_indexer_cluster = StringIndexer(inputCol="customer_state", outputCol="customer_state_idx", handleInvalid="keep")
state_encoder_cluster = OneHotEncoder(inputCols=["customer_state_idx"], outputCols=["customer_state_ohe"], handleInvalid="keep")
assembler_cluster = VectorAssembler(inputCols=numeric_cols_cluster + ["customer_state_ohe"], outputCol="raw_features", handleInvalid="keep")
scaler_cluster = StandardScaler(inputCol="raw_features", outputCol="features", withMean=False, withStd=True)

clustering_pipeline = Pipeline(stages=[state_indexer_cluster, state_encoder_cluster, assembler_cluster, scaler_cluster])
clustering_pipeline_model = clustering_pipeline.fit(cluster_train_df)
clustering_train_df = clustering_pipeline_model.transform(cluster_train_df).select("customer_unique_id", "features")
clustering_val_df = clustering_pipeline_model.transform(cluster_val_df).select("customer_unique_id", "features")
clustering_test_df = clustering_pipeline_model.transform(cluster_test_df).select("customer_unique_id", "features")
clustering_fe_df = clustering_pipeline_model.transform(cluster_df).select("customer_unique_id", "features")
clustering_fe_df.show(5, truncate=False)
print(f"clustering split rows -> train: {clustering_train_df.count():,} | val: {clustering_val_df.count():,} | test: {clustering_test_df.count():,}")
print(f"clustering_fe rows: {clustering_fe_df.count():,}")


+--------------------------------+-----------------------------------------------------------------------------------------------------------------+
|customer_unique_id              |features                                                                                                         |
+--------------------------------+-----------------------------------------------------------------------------------------------------------------+
|0000b849f77a49e4a4ce2b2a4ca5be3f|(32,[0,1,2,3,4],[1.0689819785367198,4.836339942665465,0.11942806497770632,1.9283528328329231,2.026777734004676]) |
|0000f46a3911fa3c0805444483337064|(32,[0,1,2,3,9],[3.8430885854142196,4.836339942665465,0.37870863414409117,1.9283528328329231,5.322919964513865]) |
|00053a61a98854899e70ed204dd4bafe|(32,[0,1,2,3,8],[1.5149376505643084,4.836339942665465,1.8411863287000712,3.8567056656658463,4.540456120881903])  |
|000949456b182f53c18b68d6babc79c1|(32,[0,1,2,3,4],[1.1607963816012232,4.836339942665465,0.3603925241419935

**[C6-1] Làm sạch dữ liệu ALS — filter null & invalid ratings**

- Filter bỏ các dòng null `customer_unique_id` hoặc `product_id` (không thể map sang index).
- `implicit_rating`: ép kiểu double, fill null bằng 0.0, sau đó filter `> 0`.
  Lý do: ALS cần rating dương; rating = 0 nghĩa là không có tương tác thực sự.


In [38]:
# Command 31 - ALS input preparation
als_clean_df = (
    als_base_df
    .filter(F.col("customer_unique_id").isNotNull() & F.col("product_id").isNotNull())
    .withColumn("implicit_rating", F.coalesce(F.col("implicit_rating").cast("double"), F.lit(0.0)))
    .filter(F.col("implicit_rating") > 0)
)


**[C6-2] StringIndexer cho User & Item**

- ALS (Matrix Factorization) yêu cầu `user_idx` và `item_idx` là **số nguyên** liên tục.
- `StringIndexer` với `handleInvalid='skip'`: loại bỏ user/item chưa thấy khi inference
  (thay vì crash) — cách xử lý cold-start an toàn ở bước FE.
- Fit user_indexer trước → transform → fit item_indexer trên kết quả đã có user_idx.
  Thứ tự này đảm bảo index nhất quán giữa các lần chạy.


In [39]:

# Drop existing user_idx and item_idx columns if they exist
als_index_input_df = als_clean_df.drop("user_idx", "item_idx")

user_indexer = StringIndexer(inputCol="customer_unique_id", outputCol="user_idx", handleInvalid="skip")
item_indexer = StringIndexer(inputCol="product_id", outputCol="item_idx", handleInvalid="skip")
user_indexer_model = user_indexer.fit(als_index_input_df)
als_indexed_df = user_indexer_model.transform(als_index_input_df)
item_indexer_model = item_indexer.fit(als_indexed_df)
als_indexed_df = item_indexer_model.transform(als_indexed_df)


26/04/03 00:00:19 WARN DAGScheduler: Broadcasting large task binary with size 5.4 MiB
26/04/03 00:00:20 WARN DAGScheduler: Broadcasting large task binary with size 5.4 MiB


**[C6-3] Format cuối & Train/Val/Test split cho ALS**

- Cast `user_idx`, `item_idx` về `IntegerType` (ALS MLlib yêu cầu int, không phải double).
- Rename `implicit_rating` → `rating` để khớp tên cột mặc định của ALS.
- **Hash-based split** (thay vì `randomSplit`):
  - `split_bucket = abs(hash(user_idx + '_' + item_idx)) % 20`
  - bucket < 14 → train (70%), 14–16 → val (15%), 17–19 → test (15%).
  - Lý do dùng hash: **deterministic** qua các lần chạy, không cần seed — đảm bảo
    cùng một user-item pair luôn rơi vào cùng split.


In [40]:

als_ready_df = (
    als_indexed_df
    .withColumn("user_idx", F.col("user_idx").cast("int"))
    .withColumn("item_idx", F.col("item_idx").cast("int"))
    .withColumnRenamed("implicit_rating", "rating")
    .select("customer_unique_id", "product_id", "user_idx", "item_idx", "rating")
)

als_ready_df = als_ready_df.withColumn("split_bucket", (F.abs(F.hash(F.concat_ws("_", F.col("user_idx").cast("string"), F.col("item_idx").cast("string")))) % 20).cast("int"))
als_train_df = als_ready_df.filter(F.col("split_bucket") < 14).drop("split_bucket")
als_val_df = als_ready_df.filter((F.col("split_bucket") >= 14) & (F.col("split_bucket") < 17)).drop("split_bucket")
als_test_df = als_ready_df.filter(F.col("split_bucket") >= 17).drop("split_bucket")
print(f"als_ready rows: {als_ready_df.count():,}")
print(f"als_train rows: {als_train_df.count():,} | als_val rows: {als_val_df.count():,} | als_test rows: {als_test_df.count():,}")
als_ready_df.show(5, truncate=False)


26/04/03 00:00:20 WARN DAGScheduler: Broadcasting large task binary with size 7.1 MiB
26/04/03 00:00:20 WARN DAGScheduler: Broadcasting large task binary with size 7.7 MiB


als_ready rows: 101,987


26/04/03 00:00:20 WARN DAGScheduler: Broadcasting large task binary with size 7.7 MiB
26/04/03 00:00:20 WARN DAGScheduler: Broadcasting large task binary with size 7.7 MiB


als_train rows: 71,277 | als_val rows: 15,368 | als_test rows: 15,342
+--------------------------------+--------------------------------+--------+--------+------------------+------------+
|customer_unique_id              |product_id                      |user_idx|item_idx|rating            |split_bucket|
+--------------------------------+--------------------------------+--------+--------+------------------+------------+
|f139f07dd192b4c3a1b21082a9c632a2|d9bdf643d95cb89844c0da1a0df1d16e|90238   |769     |5.795790545596741 |16          |
|9392534851a5f1f12396b5f897f9f2ad|ec02a5d380128f7a188e9ce8f3ddd832|57325   |1120    |5.8895969657192   |4           |
|5c96a427e784b5e917f5a4fbf0f9e84a|500870614ddcf5bd84f7d26861026c8a|38175   |290     |3.70805020110221  |14          |
|1c83b776b851b057dcc32bb2fa5edbdd|4bc3ab2ff016887e619e587a993af3bf|15566   |9478    |6.438079308923196 |14          |
|8797820f6554668a5b835ff74cd077df|381169576b9f3c083a1c3261da161a3b|53184   |2004    |4.8682800824802275|

26/04/03 00:00:20 WARN DAGScheduler: Broadcasting large task binary with size 7.7 MiB


**[C7] Chuẩn hóa basket input cho FP-Growth**

- `F.expr("filter(items, x -> x is not null)")`: dùng Spark SQL filter để loại bỏ
  null khỏi array (tương đương list comprehension nhưng chạy distributed).
- `basket_size = F.size(items)`: đếm số sản phẩm trong giỏ.
- `filter(basket_size >= 2)`: FP-Growth cần ít nhất 2 item để tạo ra association rule;
  giỏ 1 item không sinh được pair nào có nghĩa.
- `basket_profile_df`: phân phối basket_size — thông tin quan trọng để chọn
  `minSupport` phù hợp (quá cao → mất rule, quá thấp → rule rác).


In [41]:
# Command 32 - FP-Growth input preparation
fpgrowth_ready_df = (
    fpgrowth_base_df
    .withColumn("items", F.expr("filter(items, x -> x is not null)"))
    .withColumn("basket_size", F.size(F.col("items")))
    .filter(F.col("basket_size") >= 2)
    .select("order_id", "items", "basket_size")
)

basket_profile_df = fpgrowth_ready_df.groupBy("basket_size").count().orderBy(F.asc("basket_size"))
fpgrowth_ready_df.show(5, truncate=False)
basket_profile_df.show(20, truncate=False)


+--------------------------------+------------------------------------------------------------------------------------------------------+-----------+
|order_id                        |items                                                                                                 |basket_size|
+--------------------------------+------------------------------------------------------------------------------------------------------+-----------+
|002f98c0f7efd42638ed6100ca699b42|[d41dc2f2979f52d75d78714b378d4068, 880be32f4db1d9f6e2bec38fb6ac23ab]                                  |2          |
|005d9a5423d47281ac463a968b3936fb|[fb7a100ec8c7b34f60cec22b1a9a10e0, 4c3ae5db49258df0784827bdacf3b396]                                  |2          |
|00946f674d880be1f188abc10ad7cf46|[4dcb49b9ca7e48d2f108d40caa77caa2, 9bb2d066e4b33b624cbdfec7d50b3dcb]                                  |2          |
|01235dc626dcf13283207ba7f36a959a|[176faef125e3675b4eeaa6083ada61e1, 10dae91e0aba95747e90280edeffe88

**[C8-1] Train/Val/Test split cho Classification & Regression**

- Notebook này **reuse split đã được materialize upstream** trong `data/processed/gold/`.
- Cấu trúc hiện tại: `classification_base_{train,val,test}` và `regression_base_{train,val,test}`.
- Cách này giữ cho feature notebook không phải tự chia lại dữ liệu, tránh lệch split giữa các notebook.
- Pipeline vẫn chỉ `fit` trên train split, sau đó transform cho val/test để đảm bảo không leakage.


In [42]:
# Command 33 - Luu feature outputs va artifacts
print("Classification/Regression splits reused from pre-fit base split (fit on train only).")


Classification/Regression splits reused from pre-fit base split (fit on train only).


**[C8-2] Gom tất cả feature datasets và ghi parquet**

- `feature_outputs`: dict ánh xạ tên → DataFrame cho các datasets cần materialize
  (full FE + train/val/test splits cho classification, regression, clustering, ALS; và fpgrowth full).
- Vòng `for`: ghi tất cả dưới `data/processed/features/` với `mode('overwrite')` — idempotent.
- **Tách classification/regression thành 3 split** để training notebook chỉ cần
  `spark.read.parquet(...)` mà không cần biết logic split.


In [43]:
feature_outputs = {
    "classification_fe": classification_fe_df,
    "classification_train": classification_train_df,
    "classification_val": classification_val_df,
    "classification_test": classification_test_df,
    "regression_fe": regression_fe_df,
    "regression_train": regression_train_df,
    "regression_val": regression_val_df,
    "regression_test": regression_test_df,
    "clustering_fe": clustering_fe_df,
    "clustering_train": clustering_train_df,
    "clustering_val": clustering_val_df,
    "clustering_test": clustering_test_df,
    "als_ready": als_ready_df,
    "als_train": als_train_df,
    "als_val": als_val_df,
    "als_test": als_test_df,
    "fpgrowth_ready": fpgrowth_ready_df,
}

for dataset_name, df in feature_outputs.items():
    df.write.mode("overwrite").parquet(str(FEATURE_DIR / dataset_name))


26/04/03 00:00:22 WARN DAGScheduler: Broadcasting large task binary with size 7.9 MiB
26/04/03 00:00:22 WARN DAGScheduler: Broadcasting large task binary with size 7.9 MiB
26/04/03 00:00:23 WARN DAGScheduler: Broadcasting large task binary with size 7.9 MiB
26/04/03 00:00:23 WARN DAGScheduler: Broadcasting large task binary with size 7.9 MiB


**[C8-3] Lưu Pipeline model artifacts & in summary**

- Lưu 5 model artifacts xuống `data/processed/features/models/`:

| Artifact                     | Dùng để                                                   |
| ---------------------------- | --------------------------------------------------------- |
| `classification_fe_pipeline` | Transform data mới khi inference/UI                       |
| `regression_fe_pipeline`     | Idem cho regression                                       |
| `clustering_fe_pipeline`     | Idem cho clustering                                       |
| `als_user_indexer`           | Map customer_unique_id → user_idx cho cold-start handling |
| `als_item_indexer`           | Map product_id → item_idx                                 |

- `feature_output_summary_pdf`: bảng kiểm tra cuối — row count của các dataset output
  để xác nhận không có dataset nào bị rỗng.


In [44]:

classification_pipeline_model.write().overwrite().save(str(MODELS_DIR / "classification_fe_pipeline"))
regression_pipeline_model.write().overwrite().save(str(MODELS_DIR / "regression_fe_pipeline"))
clustering_pipeline_model.write().overwrite().save(str(MODELS_DIR / "clustering_fe_pipeline"))
user_indexer_model.write().overwrite().save(str(MODELS_DIR / "als_user_indexer"))
item_indexer_model.write().overwrite().save(str(MODELS_DIR / "als_item_indexer"))

feature_output_summary_pdf = pd.DataFrame(
    [(name, str(FEATURE_DIR / name), df.count()) for name, df in feature_outputs.items()],
    columns=["dataset_name", "output_path", "row_count"],
).sort_values("dataset_name").reset_index(drop=True)
print(feature_output_summary_pdf.to_string(index=False))


26/04/03 00:00:24 WARN TaskSetManager: Stage 269 contains a task of very large size (3766 KiB). The maximum recommended task size is 1000 KiB.
26/04/03 00:00:24 WARN TaskSetManager: Stage 273 contains a task of very large size (1306 KiB). The maximum recommended task size is 1000 KiB.
26/04/03 00:00:25 WARN DAGScheduler: Broadcasting large task binary with size 7.1 MiB
26/04/03 00:00:25 WARN DAGScheduler: Broadcasting large task binary with size 7.7 MiB
26/04/03 00:00:25 WARN DAGScheduler: Broadcasting large task binary with size 7.7 MiB
26/04/03 00:00:25 WARN DAGScheduler: Broadcasting large task binary with size 7.7 MiB


        dataset_name                                                                                                     output_path  row_count
           als_ready            /Users/thuannguyen/Downloads/Vannhi/Nhom03_BigDataCuoiKy_Update_01/data/processed/features/als_ready     101987
            als_test             /Users/thuannguyen/Downloads/Vannhi/Nhom03_BigDataCuoiKy_Update_01/data/processed/features/als_test      15342
           als_train            /Users/thuannguyen/Downloads/Vannhi/Nhom03_BigDataCuoiKy_Update_01/data/processed/features/als_train      71277
             als_val              /Users/thuannguyen/Downloads/Vannhi/Nhom03_BigDataCuoiKy_Update_01/data/processed/features/als_val      15368
   classification_fe    /Users/thuannguyen/Downloads/Vannhi/Nhom03_BigDataCuoiKy_Update_01/data/processed/features/classification_fe      98167
 classification_test  /Users/thuannguyen/Downloads/Vannhi/Nhom03_BigDataCuoiKy_Update_01/data/processed/features/classification_test    

**[C9] Training Input Contract — bảng ánh xạ model → dataset & contract cột**

Bảng tổng hợp này giải quyết một vấn đề phổ biến trong project multi-model:
mỗi family có input schema khác nhau và rất dễ nhầm.

| Model Family   | Label col          | Feature col                     | Input format           |
| -------------- | ------------------ | ------------------------------- | ---------------------- |
| Classification | `label` (int)      | `features` (Vector)             | train/val/test parquet |
| Regression     | `label` (double)   | `features` (Vector)             | train/val/test parquet |
| Clustering     | _(không có label)_ | `features` (Vector)             | train/val/test parquet |
| ALS            | `rating` (double)  | `(user_idx, item_idx)` int pair | train/val/test parquet |
| FP-Growth      | _(không có label)_ | `items` (ArrayType)             | full basket parquet    |

- In `training_input_contract_pdf` làm **checklist** trước khi sang notebook training.
- Trainer script hoặc Streamlit UI có thể đọc bảng này để auto-configure input paths.


In [45]:
# Command 34 - Chot metadata contract cho training
training_input_contract_pdf = pd.DataFrame(
    [
        ("classification", "features/classification_train + features/classification_val + features/classification_test", "label", "features", "LR, RF, NaiveBayes, LinearSVC, GBTClassifier"),
        ("regression", "features/regression_train + features/regression_val + features/regression_test", "label", "features", "LinearRegression, DecisionTreeRegressor, RandomForestRegressor"),
        ("clustering", "features/clustering_train + features/clustering_val + features/clustering_test", None, "features", "KMeans, BisectingKMeans, GaussianMixture"),
        ("als", "features/als_train + features/als_val + features/als_test", "rating", "(user_idx, item_idx)", "ALS implicit feedback"),
        ("fp_growth", "features/fpgrowth_ready", None, "items", "FPGrowth + AssociationRules"),
    ],
    columns=["model_family", "dataset_path", "label_col", "feature_col_contract", "algorithms"],
).sort_values("model_family").reset_index(drop=True)
print(training_input_contract_pdf.to_string(index=False))


  model_family                                                                               dataset_path label_col feature_col_contract                                                     algorithms
           als                                  features/als_train + features/als_val + features/als_test    rating (user_idx, item_idx)                                          ALS implicit feedback
classification features/classification_train + features/classification_val + features/classification_test     label             features                   LR, RF, NaiveBayes, LinearSVC, GBTClassifier
    clustering             features/clustering_train + features/clustering_val + features/clustering_test      None             features                       KMeans, BisectingKMeans, GaussianMixture
     fp_growth                                                                    features/fpgrowth_ready      None                items                                    FPGrowth + AssociationRules
